# Comptage des occurrences des personnages par tome

Ce notebook :

1. lit le fichier `.ods` des personnages ;
2. construit, pour chaque personnage, une liste de variantes de noms à partir des **3 premières colonnes** :
   - `NOMS HUMAINS`
   - `SECOND NOM`
   - `TROISIEME NOM`
3. prend en compte les combinaisons demandées :
   - `1 + 2`
   - `2 + 3`
   - `1 + 2 + 3`
   - `1 + 3`
4. compte les occurrences de chaque personnage dans chaque **tome** (`book`) du fichier `.csv` ;
5. exporte un tableau final qui reprend la structure du tableau `.ods` et ajoute une colonne par tome.

## Remarque importante
Par défaut, le notebook compte aussi les **formes simples** (`1`, `2`, `3`) quand elles existent.  
C'est pratique pour éviter que les personnages à nom simple aient toujours `0`.

Si tu veux **strictement** ne compter que les combinaisons demandées, mets :

```python
INCLUDE_SINGLE_NAMES = False
```
dans la cellule de paramètres.


In [1]:
# Paramètres
ODS_PATH = "Noms personnages de Pern Noms de famille.ods"
CSV_PATH = "La-ballade-de-Pern-intégrale-sentences-lemma.csv"
ODS_SHEET = "Nom et genre persos"

# True = compte aussi les noms simples (recommandé)
# False = ne compte QUE 1+2, 2+3, 1+2+3, 1+3
INCLUDE_SINGLE_NAMES = True

# Colonne texte à analyser dans le CSV
TEXT_COLUMN = "text"

# Fichiers de sortie
OUTPUT_XLSX = "pern_occurrences_par_tome.xlsx"
OUTPUT_CSV = "pern_occurrences_par_tome.csv"
OUTPUT_ALIASES_CSV = "pern_aliases_utilises.csv"


In [2]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

/home/marc-at/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/marc-at/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [3]:
# Lecture des fichiers
names_df = pd.read_excel(ODS_PATH, sheet_name=ODS_SHEET, engine="odf")
text_df = pd.read_csv(CSV_PATH)

print("Dimensions ODS :", names_df.shape)
print("Dimensions CSV :", text_df.shape)
print("\nColonnes ODS :", list(names_df.columns))
print("\nColonnes CSV :", list(text_df.columns))

Dimensions ODS : (1067, 7)
Dimensions CSV : (132850, 7)

Colonnes ODS : ['NOMS HUMAINS', 'SECOND NOM', 'TROISIEME NOM', 'NOMS DRAGONS', 'COULEUR DRAGON', 'H/F/O', 'Id unique']

Colonnes CSV : ['sentence_id', 'book', 'parution', 'start_pdf_page', 'end_pdf_page', 'text', 'text_lemma']


In [4]:
# Vérifications minimales
required_name_cols = ["NOMS HUMAINS", "SECOND NOM", "TROISIEME NOM", "Id unique"]
required_csv_cols = ["book", TEXT_COLUMN]

missing_ods = [c for c in required_name_cols if c not in names_df.columns]
missing_csv = [c for c in required_csv_cols if c not in text_df.columns]

if missing_ods:
    raise ValueError(f"Colonnes manquantes dans le ODS : {missing_ods}")

if missing_csv:
    raise ValueError(f"Colonnes manquantes dans le CSV : {missing_csv}")

print("Vérification OK.")

Vérification OK.


In [5]:
def clean_part(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None


def build_aliases(row, include_single_names=True):
    """Construit les variantes d'un personnage à partir des 3 premières colonnes."""
    p1 = clean_part(row["NOMS HUMAINS"])
    p2 = clean_part(row["SECOND NOM"])
    p3 = clean_part(row["TROISIEME NOM"])

    aliases = []

    # Optionnel : formes simples
    if include_single_names:
        for p in (p1, p2, p3):
            if p:
                aliases.append(p)

    # Combinaisons demandées
    if p1 and p2:
        aliases.append(f"{p1} {p2}")      # 1 + 2
    if p2 and p3:
        aliases.append(f"{p2} {p3}")      # 2 + 3
    if p1 and p2 and p3:
        aliases.append(f"{p1} {p2} {p3}") # 1 + 2 + 3
    if p1 and p3:
        aliases.append(f"{p1} {p3}")      # 1 + 3

    # Déduplication en conservant l'ordre
    seen = set()
    clean_aliases = []
    for a in aliases:
        a = re.sub(r"\s+", " ", a).strip()
        key = a.lower()
        if key not in seen:
            seen.add(key)
            clean_aliases.append(a)

    # Important : les alias longs d'abord pour éviter le double comptage
    clean_aliases = sorted(clean_aliases, key=lambda x: (-len(x), x.lower()))
    return clean_aliases


names_df["aliases"] = names_df.apply(
    lambda row: build_aliases(row, include_single_names=INCLUDE_SINGLE_NAMES),
    axis=1
)

names_df[["NOMS HUMAINS", "SECOND NOM", "TROISIEME NOM", "Id unique", "aliases"]].head(15)

,NOMS HUMAINS,SECOND NOM,TROISIEME NOM,Id unique,aliases
0,Kenjo,Fusaiyuki,NaN,1.0,"[Kenjo Fusaiyuki, Fusaiyuki, Kenjo]"
1,Abula,NaN,NaN,2.0,[Abula]
2,Abuna,NaN,NaN,3.0,[Abuna]
3,Adessa,NaN,NaN,4.0,[Adessa]
4,Adrea,Beljeth,NaN,5.0,"[Adrea Beljeth, Beljeth, Adrea]"
5,Aida,NaN,NaN,6.0,[Aida]
6,Alda,NaN,NaN,7.0,[Alda]
7,Alianne,Zulueta,NaN,8.0,"[Alianne Zulueta, Alianne, Zulueta]"
8,Aliena,Zuleita,NaN,9.0,"[Aliena Zuleita, Zuleita, Aliena]"
9,Almie,NaN,NaN,10.0,[Almie]


In [6]:
# Tableau d'audit des alias réellement utilisés
aliases_audit = names_df[["Id unique", "NOMS HUMAINS", "SECOND NOM", "TROISIEME NOM", "aliases"]].copy()
aliases_audit["aliases"] = aliases_audit["aliases"].apply(lambda x: " | ".join(x))
aliases_audit.to_csv(OUTPUT_ALIASES_CSV, index=False)
print(f"Fichier alias exporté : {OUTPUT_ALIASES_CSV}")
aliases_audit.head(10)

Fichier alias exporté : pern_aliases_utilises.csv


,Id unique,NOMS HUMAINS,SECOND NOM,TROISIEME NOM,aliases
0,1.0,Kenjo,Fusaiyuki,NaN,Kenjo Fusaiyuki | Fusaiyuki | Kenjo
1,2.0,Abula,NaN,NaN,Abula
2,3.0,Abuna,NaN,NaN,Abuna
3,4.0,Adessa,NaN,NaN,Adessa
4,5.0,Adrea,Beljeth,NaN,Adrea Beljeth | Beljeth | Adrea
5,6.0,Aida,NaN,NaN,Aida
6,7.0,Alda,NaN,NaN,Alda
7,8.0,Alianne,Zulueta,NaN,Alianne Zulueta | Alianne | Zulueta
8,9.0,Aliena,Zuleita,NaN,Aliena Zuleita | Zuleita | Aliena
9,10.0,Almie,NaN,NaN,Almie


In [7]:
# Préparation des textes par tome
text_df = text_df.copy()
text_df = text_df[text_df["book"].notna()].copy()
text_df["book"] = text_df["book"].astype(int)

books = sorted(text_df["book"].unique())
print("Tomes détectés :", books)

book_texts = (
    text_df.groupby("book")[TEXT_COLUMN]
    .apply(lambda s: "\n".join(s.fillna("").astype(str)))
    .to_dict()
)

for b in books[:5]:
    print(f"Tome {b}: {len(book_texts[b]):,} caractères")

Tomes détectés : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Tome 1: 777,188 caractères
Tome 2: 562,067 caractères
Tome 3: 634,318 caractères
Tome 4: 727,919 caractères
Tome 5: 240,296 caractères


In [8]:
def compile_character_pattern(aliases):
    """Crée une regex qui repère les alias du personnage avec bornes de mots."""
    if not aliases:
        return None

    escaped = [re.escape(a) for a in aliases]
    pattern = r"(?<!\w)(?:" + "|".join(escaped) + r")(?!\w)"
    return re.compile(pattern, flags=re.IGNORECASE)


def count_occurrences_in_text(pattern, text):
    if pattern is None:
        return 0
    return len(pattern.findall(text))

In [9]:
# Comptage des occurrences par personnage et par tome
count_data = defaultdict(list)

for i, row in names_df.iterrows():
    aliases = row["aliases"]
    pattern = compile_character_pattern(aliases)

    for book in books:
        n = count_occurrences_in_text(pattern, book_texts[book])
        count_data[f"book_{book}"].append(n)

    if (i + 1) % 100 == 0:
        print(f"{i + 1} personnages traités / {len(names_df)}")

print("Comptage terminé.")

100 personnages traités / 1067
200 personnages traités / 1067
300 personnages traités / 1067
400 personnages traités / 1067
500 personnages traités / 1067
600 personnages traités / 1067
700 personnages traités / 1067
800 personnages traités / 1067
900 personnages traités / 1067
1000 personnages traités / 1067
Comptage terminé.


In [10]:
# Assemblage du tableau final
output_df = names_df.drop(columns=["aliases"]).copy()

for col_name, values in count_data.items():
    output_df[col_name] = values

# Optionnel : total tous tomes confondus
book_cols = [f"book_{b}" for b in books]
output_df["total_occurrences"] = output_df[book_cols].sum(axis=1)

print(output_df.shape)
output_df.head(10)

(1067, 23)


,NOMS HUMAINS,SECOND NOM,TROISIEME NOM,NOMS DRAGONS,COULEUR DRAGON,H/F/O,Id unique,book_1,book_2,book_3,...,book_7,book_8,book_9,book_10,book_11,book_12,book_13,book_14,book_15,total_occurrences
0,Kenjo,Fusaiyuki,NaN,NaN,NaN,Cyborg,1.0,175,0,0,...,0,0,0,0,0,0,0,2,11,188
1,Abula,NaN,NaN,NaN,NaN,FEMME,2.0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
2,Abuna,NaN,NaN,NaN,NaN,FEMME,3.0,0,0,0,...,0,0,0,0,20,3,0,0,0,23
3,Adessa,NaN,NaN,NaN,NaN,FEMME,4.0,0,0,0,...,4,0,0,0,0,0,0,0,0,4
4,Adrea,Beljeth,NaN,NaN,NaN,FEMME,5.0,0,4,0,...,0,0,0,0,0,0,0,2,11,22
5,Aida,NaN,NaN,NaN,NaN,FEMME,6.0,7,4,2,...,5,1,4,4,1,5,5,8,1,68
6,Alda,NaN,NaN,NaN,NaN,FEMME,7.0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
7,Alianne,Zulueta,NaN,NaN,NaN,FEMME,8.0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
8,Aliena,Zuleita,NaN,NaN,NaN,FEMME,9.0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
9,Almie,NaN,NaN,NaN,NaN,FEMME,10.0,0,6,0,...,0,0,0,0,0,0,0,0,0,6


In [11]:
# Export
output_df.to_excel(OUTPUT_XLSX, index=False)
output_df.to_csv(OUTPUT_CSV, index=False)

print(f"Fichier Excel exporté : {OUTPUT_XLSX}")
print(f"Fichier CSV exporté   : {OUTPUT_CSV}")

Fichier Excel exporté : pern_occurrences_par_tome.xlsx
Fichier CSV exporté   : pern_occurrences_par_tome.csv
